# Static Aerodynamic Derivatives in VLMPy

This notebook explains and verifies the actual `VLM.calculate_static_derivatives` implementation. It builds on [Aerodynamic Coefficients](06_aerodynamic_coefficients.ipynb), covering the coefficient definitions and panel loads in more detail.

**Scope:** steady, incompressible, inviscid, irrotational flow, thin lifting surfaces, attached flow, and a prescribed/linearized wake. The derivatives are local finite-difference sensitivities of the current steady implementation, not experimentally validated aircraft stability data. They do not model viscosity, stall, compressibility, unsteady response, angular-rate derivatives ($p$, $q$, $r$), trim, a center of gravity, or flight dynamics.

## Theory: What Is a Static Aerodynamic Derivative?

A static aerodynamic derivative is the local slope of a dimensionless force or moment coefficient with respect to a small change in an operating variable, with the other variables held fixed. For example, $C_{Z_\alpha}=\partial C_Z/\partial\alpha$ measures the change in the solver's $z$-force coefficient per radian of angle of attack; $C_{m_\alpha}=\partial C_m/\partial\alpha$ measures the corresponding change in pitching-moment coefficient. Similarly, derivatives with respect to sideslip $\beta$ describe lateral-directional response, while derivatives with respect to a control deflection $\delta$ describe control effectiveness. Uppercase $C_L$ is lift; lowercase $C_l$ denotes rolling moment.

Near a reference condition, the first-order approximation is

$$C\approx C_0+C_\alpha\Delta\alpha+C_\beta\Delta\beta+C_\delta\Delta\delta.$$

A derivative describes a **local slope**, not a coefficient at a different flight condition; the approximation becomes less reliable as the perturbations grow or nonlinear effects become important. These are partial derivatives with fixed controls and other inputs, not total derivatives along a trim curve. They are static sensitivities, distinct from damping derivatives with respect to angular rates or unsteady motion.

In a conventional nose-up-positive pitching convention, a negative $C_{m_\alpha}$ about a **fixed center of gravity** is a standard criterion for a restoring static pitch response near trim. VLMPy's raw `dCm_dalpha` uses its own axis signs and a reference point that can move during perturbation, and its baseline is not necessarily trimmed. Its sign alone cannot establish aircraft stability. See [Aircraft Flight Mechanics: Longitudinal Stability](https://aircraftflightmechanics.com/StaticStability/LSS.html) and the reference-point discussion below.

## 1. Baseline Case

A symmetric rectangular NACA 0012 wing has $b=8$ m, $c=1$ m, $S=8$ m², $U=40$ m/s, and $\rho=1.225$ kg/m³. The operating point is $\alpha=2^\circ$, $\beta=0$, with a symmetric wing flap at $\delta=0$. The wing spans $y=-4$ to $+4$ m; the flap covers 60–100% of each semispan and starts at 70% of local chord. `n=16` is the full-span panel count and `m=4` the chordwise count. This operating point is not asserted to be trimmed.

The following cell calls the real VLMPy solver and its derivative API. Angles supplied to `VLM` and stored geometry are radians; `step_deg` is specified in degrees, while returned derivatives are **per radian**.

## How the Code Is Used

Run the cells sequentially from a repository directory. The baseline calls `VLM.run_vlm()` and then `calculate_static_derivatives(step_deg=0.1)`. The returned dictionaries contain force and moment coefficient derivatives with respect to angle of attack, sideslip, and configured wing controls. Subsequent examples reconstruct central differences with separate cases, translate moments to a fixed coordinate, examine mesh and step sensitivity, and compare a scalar lift slope with an independent lifting-line model.

The central-difference calculation is a software verification of the same solver, while theoretical comparison and experimental validation are distinct activities. Reference axes, control signs, reference dimensions, and the moment point must be matched before using a derivative in another model.

In [1]:
import contextlib
import copy
import io
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def find_repository(start):
    for directory in (start, *start.parents):
        if (directory / 'src' / 'vlm').is_dir():
            return directory
    raise RuntimeError('Could not locate the repository directory src/vlm.')


ROOT = find_repository(Path.cwd().resolve())
for directory in (ROOT, ROOT / 'src' / 'vlm'):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from src.vlm.lib.vlm import VLM


COEFFICIENTS = ('CX', 'CY', 'CZ', 'Cl', 'Cm', 'Cn')
wing = {'wing_sections': [{
    'span_fraction': 8.0, 'chord_root': 1.0, 'chord_tip': 1.0,
    'sweep': 0.0, 'dihedral': 0.0, 'NACA_root': '0012', 'NACA_tip': '0012',
    'flap_start': 0.6, 'flap_end': 1.0, 'flap_hinge_chord': 0.7,
    'deflection_angle': 0.0, 'deflection_type': 'symmetrical',
}]}


def solve_case(plane, alpha_deg=2.0, beta_deg=0.0, n=16, m=4, speed=40.0, density=1.225):
    case = VLM(copy.deepcopy(plane), speed, density, np.deg2rad(alpha_deg), np.deg2rad(beta_deg), n, m)
    with contextlib.redirect_stdout(io.StringIO()):
        case.run_vlm()
    return case


case = solve_case(wing)
with contextlib.redirect_stdout(io.StringIO()):
    baseline_derivatives = case.calculate_static_derivatives(step_deg=0.1)
assert baseline_derivatives['units'] == 'per radian'
assert set(baseline_derivatives['controls']) == {'wing_section_1'}
assert np.isfinite(list(baseline_derivatives['alpha'].values())).all()
print(f"Baseline: CL={case.CL:.6f}, CD={case.CD:.6f}, dCZ/dalpha={baseline_derivatives['alpha']['dCZ_dalpha']:.6f} rad⁻¹")

Baseline: CL=0.165934, CD=0.001182, dCZ/dalpha=4.760686 rad⁻¹


## 2. Definition and Units

For coefficient $C$ and parameter $x\in\{\alpha,\beta,\delta\}$, the local partial derivative holds all other inputs fixed:

$$C_{x}\equiv\left.\frac{\partial C}{\partial x}\right|_0
\approx\frac{C(x_0+h)-C(x_0-h)}{2h},\qquad h=\mathrm{step\_deg}\,\frac{\pi}{180}.$$

A dimensionless coefficient divided by an angle in radians gives units rad$^{-1}$. A derivative per degree is the per-radian value multiplied by $\pi/180$. Central differences have a nominal $O(h^2)$ truncation error for a sufficiently smooth function; excessively small steps can amplify floating-point and solver noise.

The API returns six derivatives each for `alpha` and `beta`: `CX`, `CY`, `CZ`, `Cl`, `Cm`, and `Cn`. `controls` contains six derivatives per wing section that has `deflection_angle`, with keys `wing_section_1`, etc.; the derivative is with respect to that section's configured deflection. It does **not** automatically add derivatives for stabilizer controls or for sections without `deflection_angle`. Each ± evaluation rebuilds the state, and the original plane, angles, mesh, solved coefficients, and `results` are restored afterwards.

In [2]:
print(f"Step: {baseline_derivatives['step_deg']:.3f}° = {np.deg2rad(baseline_derivatives['step_deg']):.8f} rad")
print(f"{'Coefficient':12s} {'d/dalpha':>14s} {'d/dbeta':>14s} {'d/ddelta':>14s}  [rad⁻¹]")
for coefficient in COEFFICIENTS:
    alpha_value = baseline_derivatives['alpha'][f'd{coefficient}_dalpha']
    beta_value = baseline_derivatives['beta'][f'd{coefficient}_dbeta']
    control_value = baseline_derivatives['controls']['wing_section_1'][f'd{coefficient}_ddelta']
    print(f'{coefficient:12s} {alpha_value:+14.6f} {beta_value:+14.6f} {control_value:+14.6f}')
print(f"dCZ/dalpha per degree: {baseline_derivatives['alpha']['dCZ_dalpha'] * np.pi / 180:+.7f}")

Step: 0.100° = 0.00174533 rad
Coefficient        d/dalpha        d/dbeta       d/ddelta  [rad⁻¹]
CX                -0.067786      +0.000000      -0.019487
CY                +0.000000      +0.001182      -0.000000
CZ                +4.760686      +0.000000      +1.116893
Cl                -0.000000      -0.000001      -0.000000
Cm                +0.919933      +0.000000      -0.044015
Cn                -0.000000      -0.000024      -0.000000
dCZ/dalpha per degree: +0.0830896


## 3. Axes, Reference Quantities, and Control Signs

VLMPy stores panel coordinates internally as `(y, x, z)` and reorders them to `(x, y, z)` in coefficient post-processing. At $\alpha=\beta=0$, $+x$ is along the solver's freestream, $y$ is spanwise, and $z$ is vertical; moments use $\mathbf r\times\mathbf F$ and the right-hand rule. Do not assume these signs match another tool's conventional aircraft body axes without transforming them.

The coefficients are normalized by $q_\infty S_{ref}$ for forces, $q_\infty S_{ref}b$ for roll and yaw, and $q_\infty S_{ref}\bar c$ for pitch. Reference dimensions use the main-wing sections, while contributions from defined lifting surfaces are included in the net force and moment. The moment reference is the area-weighted average of the main-wing bound-segment midpoints, **not an explicitly specified CG**. It is recalculated at each perturbed state and may move as geometry changes.

For this flap implementation, a positive symmetric deflection subtracts $\delta(x-x_{hinge})$ from the local $z$ coordinate on both sides. For an antisymmetric control, the $y>0$ side uses that sign and $y<0$ uses the opposite sign. These are implementation conventions; verify them against an external code before comparing signs.

For context on conventional moments about a fixed CG and nose-up stability conventions, see [NASA Glenn: Aircraft Rotations](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/aircraft-rotations/) and [Aircraft Flight Mechanics: Longitudinal Stability](https://aircraftflightmechanics.com/StaticStability/LSS.html). A raw VLMPy `dCm_dalpha` cannot be interpreted as a stability verdict until axes, sign, trim, and the moment reference have been reconciled.

In [3]:
reference = case.aerodynamic_coefficients['reference']
print(f"Reference S={reference['area']:.3f} m², b={reference['span']:.3f} m, c̄={reference['mean_aerodynamic_chord']:.3f} m")
print('Reference point [x, y, z] [m]:', reference['point'])
print('Baseline scalar CL and CD are distinct from the six vector coefficients.')
print('Freestream reordered to [x, y, z] [m/s]:', np.asarray(case.u_)[[1, 0, 2]])

Reference S=8.000 m², b=8.000 m, c̄=1.000 m
Reference point [x, y, z] [m]: [ 4.37233487e-01  2.77555756e-17 -1.52685298e-02]
Baseline scalar CL and CD are distinct from the six vector coefficients.
Freestream reordered to [x, y, z] [m/s]: [40.  0.  0.]


## 4. API State Restoration and Direct Finite-Difference Check

The API perturbs the existing object, solves each case, and restores the baseline in a `finally` block. First check the baseline configuration, circulation, coefficients, and `results` after the call. Then construct **separate VLM objects** for the positive and negative perturbations and compare all 18 independently assembled central differences with the returned values. This verifies the API's arithmetic and state handling; both paths still use the same aerodynamic solver and do not constitute independent physical validation.

In [4]:
original_configuration = case.export_configuration(include_flight=True)
original_gamma = case.gammas.copy()
original_coefficients = np.array([getattr(case, name) for name in COEFFICIENTS])
saved_results = {'example': 'preserved'}
case.results = saved_results
with contextlib.redirect_stdout(io.StringIO()):
    derivatives = case.calculate_static_derivatives(step_deg=0.1)
assert case.export_configuration(include_flight=True) == original_configuration
assert case.results is saved_results
assert np.allclose(case.gammas, original_gamma, rtol=1e-12, atol=1e-12)
assert np.allclose([getattr(case, name) for name in COEFFICIENTS], original_coefficients, rtol=1e-12, atol=1e-12)


def perturbed_case(parameter, change_deg, plane=wing, n=16, m=4):
    perturbed_plane = copy.deepcopy(plane)
    alpha_deg, beta_deg = 2.0, 0.0
    if parameter == 'alpha':
        alpha_deg += change_deg
    elif parameter == 'beta':
        beta_deg += change_deg
    elif parameter == 'delta':
        perturbed_plane['wing_sections'][0]['deflection_angle'] += np.deg2rad(change_deg)
    else:
        raise ValueError(f'Unsupported perturbation: {parameter}')
    return solve_case(perturbed_plane, alpha_deg=alpha_deg, beta_deg=beta_deg, n=n, m=m)


def finite_difference(parameter, step_deg=0.1, plane=wing, n=16, m=4):
    plus = perturbed_case(parameter, step_deg, plane=plane, n=n, m=m)
    minus = perturbed_case(parameter, -step_deg, plane=plane, n=n, m=m)
    delta = 2 * np.deg2rad(step_deg)
    return np.array([(getattr(plus, name) - getattr(minus, name)) / delta for name in COEFFICIENTS])


for parameter in ('alpha', 'beta', 'delta'):
    api_block = derivatives['controls']['wing_section_1'] if parameter == 'delta' else derivatives[parameter]
    api_values = np.array([api_block[f'd{name}_d{parameter}'] for name in COEFFICIENTS])
    direct_values = finite_difference(parameter)
    discrepancy = np.max(np.abs(api_values - direct_values))
    assert np.allclose(api_values, direct_values, rtol=1e-10, atol=1e-10)
    print(f'{parameter:5s}: maximum API/direct difference = {discrepancy:.3e} rad⁻¹')
print('Baseline configuration, circulation, coefficients, and results restored.')

alpha: maximum API/direct difference = 0.000e+00 rad⁻¹
beta : maximum API/direct difference = 0.000e+00 rad⁻¹
delta: maximum API/direct difference = 0.000e+00 rad⁻¹
Baseline configuration, circulation, coefficients, and results restored.


## 5. Why the Moment Reference Matters

For the resultant force $\mathbf F$, the moment about a fixed point $P$ can be obtained from the reported moment about $R$ as

$$\mathbf M_P=\mathbf M_R+(\mathbf r_R-\mathbf r_P)\times\mathbf F.$$

We choose $P$ equal to the solver's baseline reference point expressed in the post-processing coordinates, hold its numerical coordinates fixed for the two perturbations, and translate each moment before differencing. This is a **fixed-coordinate demonstration**, not a user-defined CG or an aircraft trim analysis. Because $R$ can move, the resulting derivative can differ from `dCm_dalpha` reported directly by the API. The example uses the same $S_{ref}$ and $\bar c$ for all states; the calculation is only valid when those reference dimensions are consistent.

In [5]:
fixed_point = np.asarray(case.aerodynamic_coefficients['reference']['point']).copy()
step_deg = 0.1
alpha_plus = perturbed_case('alpha', step_deg)
alpha_minus = perturbed_case('alpha', -step_deg)


def pitching_coefficient_at_fixed_point(vlm, point):
    data = vlm.aerodynamic_coefficients
    ref = data['reference']
    translated = data['moment'] + np.cross(np.asarray(ref['point']) - point, data['force'])
    dynamic_pressure = 0.5 * vlm.rho * vlm.u**2
    return translated[1] / (dynamic_pressure * ref['area'] * ref['mean_aerodynamic_chord'])


fixed_derivative = (pitching_coefficient_at_fixed_point(alpha_plus, fixed_point)
                    - pitching_coefficient_at_fixed_point(alpha_minus, fixed_point)) / (2 * np.deg2rad(step_deg))
moving_derivative = derivatives['alpha']['dCm_dalpha']
reference_shift = np.asarray(alpha_plus.aerodynamic_coefficients['reference']['point']) - np.asarray(alpha_minus.aerodynamic_coefficients['reference']['point'])
assert np.isfinite([fixed_derivative, moving_derivative]).all()
print(f'dCm/dalpha, solver reference: {moving_derivative:+.6f} rad⁻¹')
print(f'dCm/dalpha, fixed baseline point: {fixed_derivative:+.6f} rad⁻¹')
print('Reference-point change between ± perturbations [m]:', reference_shift)
print(f'Difference: {fixed_derivative - moving_derivative:+.6f} rad⁻¹')

dCm/dalpha, solver reference: +0.919933 rad⁻¹
dCm/dalpha, fixed baseline point: +0.922988 rad⁻¹
Reference-point change between ± perturbations [m]: [-5.32971964e-05  2.77555756e-17 -1.52623201e-03]
Difference: +0.003055 rad⁻¹


## 6. Step-Size and Mesh Sensitivity

A central difference is local to the operating point. Compare several angular steps at a fixed mesh, then refine the spanwise panel count while keeping the chordwise count and step fixed. A plateau in step size or a small difference between two meshes is useful evidence, but neither proves mesh convergence or physical accuracy. Near geometric discontinuities, panel-boundary effects can affect control derivatives; inspect the results rather than assuming monotonic convergence.

In [6]:
steps_deg = np.array([0.02, 0.05, 0.1, 0.2, 0.4])
step_study = []
for step in steps_deg:
    with contextlib.redirect_stdout(io.StringIO()):
        value = case.calculate_static_derivatives(step_deg=float(step))
    step_study.append({'step_deg': step, 'dCZ_dalpha': value['alpha']['dCZ_dalpha'],
                       'dCm_dalpha': value['alpha']['dCm_dalpha'],
                       'dCZ_ddelta': value['controls']['wing_section_1']['dCZ_ddelta']})

mesh_study = []
for span_panels in (8, 16, 32):
    mesh_case = solve_case(wing, n=span_panels, m=4)
    with contextlib.redirect_stdout(io.StringIO()):
        value = mesh_case.calculate_static_derivatives(step_deg=0.1)
    mesh_study.append({'n': span_panels, 'm': 4, 'dCZ_dalpha': value['alpha']['dCZ_dalpha'],
                       'dCm_dalpha': value['alpha']['dCm_dalpha'],
                       'dCZ_ddelta': value['controls']['wing_section_1']['dCZ_ddelta']})

print('Step [deg]    dCZ/dalpha    dCm/dalpha    dCZ/ddelta [rad⁻¹]')
for row in step_study:
    print(f"{row['step_deg']:10.3f} {row['dCZ_dalpha']:+13.6f} {row['dCm_dalpha']:+13.6f} {row['dCZ_ddelta']:+13.6f}")
print('\nSpanwise meshes (m=4, step=0.1°):')
for row in mesh_study:
    print(f"n={row['n']:2d}: dCZ/dalpha={row['dCZ_dalpha']:+.6f}, dCm/dalpha={row['dCm_dalpha']:+.6f}, dCZ/ddelta={row['dCZ_ddelta']:+.6f}")
assert all(np.isfinite([row['dCZ_dalpha'], row['dCm_dalpha'], row['dCZ_ddelta']]).all() for row in step_study + mesh_study)

fig_sensitivity = make_subplots(rows=1, cols=2, subplot_titles=('Finite-difference step', 'Spanwise mesh'))
for name in ('dCZ_dalpha', 'dCm_dalpha', 'dCZ_ddelta'):
    fig_sensitivity.add_trace(go.Scatter(x=steps_deg, y=[row[name] for row in step_study], mode='lines+markers', name=name), row=1, col=1)
    fig_sensitivity.add_trace(go.Scatter(x=[row['n'] for row in mesh_study], y=[row[name] for row in mesh_study], mode='lines+markers', name=name, showlegend=False), row=1, col=2)
fig_sensitivity.update_xaxes(title_text='Step [degrees]', type='log', row=1, col=1)
fig_sensitivity.update_xaxes(title_text='Spanwise panels n', row=1, col=2)
fig_sensitivity.update_yaxes(title_text='Derivative [rad⁻¹]')
fig_sensitivity.update_layout(title='Sensitivity of static derivatives', height=420)
fig_sensitivity.show()

Step [deg]    dCZ/dalpha    dCm/dalpha    dCZ/ddelta [rad⁻¹]
     0.020     +4.760679     +0.919935     +1.116893
     0.050     +4.760680     +0.919935     +1.116893
     0.100     +4.760686     +0.919933     +1.116893
     0.200     +4.760710     +0.919927     +1.116893
     0.400     +4.760806     +0.919904     +1.116895

Spanwise meshes (m=4, step=0.1°):
n= 8: dCZ/dalpha=+4.887463, dCm/dalpha=+0.939999, dCZ/ddelta=+0.991592
n=16: dCZ/dalpha=+4.760686, dCm/dalpha=+0.919933, dCZ/ddelta=+1.116893
n=32: dCZ/dalpha=+4.698340, dCm/dalpha=+0.906869, dCZ/ddelta=+0.990780


The control derivative above is not monotonic across the tested meshes. The flap is applied at mesh vertices within prescribed boundaries, so changing the mesh changes how those boundaries are represented; this is a plausible contributor, not a proven error mechanism. Do not select a single mesh result as converged. The spanwise study also holds `m=4` fixed, so it does not establish chordwise convergence.

## 7. Physical Plausibility and Theoretical Context

With a symmetric geometry, symmetric control, and $\beta=0$, a small symmetric $\alpha$ or flap perturbation should not create appreciable net lateral force, roll, or yaw in this case. Likewise, a small antisymmetric $\beta$ perturbation should have nearly zero first-order change in the symmetric components $C_X$, $C_Z$, and $C_m$. These are symmetry checks under this implementation's conventions, not universal sign rules for all aircraft.

For an ideal elliptically loaded wing, inviscid lifting-line theory gives $dC_L/d\alpha=2\pi AR/(AR+2)$ per radian when the two-dimensional slope is $2\pi$. Our rectangular wing is not elliptical and the API returns $dC_Z/d\alpha$, not $dC_L/d\alpha$. Therefore compute a separate scalar $dC_L/d\alpha$ and compare it with $dC_Z/d\alpha$ and the elliptical formula only as a **plausibility reference**, not an exact benchmark or experimental validation. See [Leishman, Lifting Line & Finite Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/), the sections on elliptical loading and lift-curve slope.

In [7]:
symmetry_components = [derivatives['alpha'][f'd{name}_dalpha'] for name in ('CY', 'Cl', 'Cn')]
symmetry_components += [derivatives['beta'][f'd{name}_dbeta'] for name in ('CX', 'CZ', 'Cm')]
symmetry_components += [derivatives['controls']['wing_section_1'][f'd{name}_ddelta'] for name in ('CY', 'Cl', 'Cn')]
max_symmetry_error = np.max(np.abs(symmetry_components))
assert max_symmetry_error < 1e-10
scalar_cl_slope = (alpha_plus.CL - alpha_minus.CL) / (2 * np.deg2rad(step_deg))
aspect_ratio = reference['span']**2 / reference['area']
elliptical_slope = 2 * np.pi * aspect_ratio / (aspect_ratio + 2)
print(f'Maximum symmetry-forbidden derivative: {max_symmetry_error:.3e} rad⁻¹')
print(f'dCL/dalpha, scalar central difference: {scalar_cl_slope:.6f} rad⁻¹')
print(f"dCZ/dalpha, vector API: {derivatives['alpha']['dCZ_dalpha']:.6f} rad⁻¹")
print(f'Ideal elliptical lifting-line slope: {elliptical_slope:.6f} rad⁻¹')
print(f'Scalar slope difference from ideal elliptical reference: {abs(scalar_cl_slope - elliptical_slope) / abs(elliptical_slope):.2%}')

Maximum symmetry-forbidden derivative: 6.809e-12 rad⁻¹
dCL/dalpha, scalar central difference: 4.760686 rad⁻¹
dCZ/dalpha, vector API: 4.760686 rad⁻¹
Ideal elliptical lifting-line slope: 5.026548 rad⁻¹
Scalar slope difference from ideal elliptical reference: 5.29%


## 8. Using the Derivatives in a Local Linear Approximation

A first-order approximation about the baseline is

$$C(\alpha_0+\Delta\alpha,\beta_0+\Delta\beta,\delta_0+\Delta\delta)\approx C_0+C_\alpha\Delta\alpha+C_\beta\Delta\beta+C_\delta\Delta\delta.$$

All increments below are converted to radians. The comparison with a separate combined-perturbation solve reveals local linearization error for this one case; it is not a flight-dynamics model or a guarantee outside the tested neighborhood. It also retains the solver's moving moment-reference convention.

In [8]:
increments_deg = {'alpha': 0.3, 'beta': 0.2, 'delta': 0.2}
combined_wing = copy.deepcopy(wing)
combined_wing['wing_sections'][0]['deflection_angle'] += np.deg2rad(increments_deg['delta'])
combined = solve_case(combined_wing, alpha_deg=2.0 + increments_deg['alpha'], beta_deg=increments_deg['beta'])
print(f"{'Coefficient':12s} {'Linear':>12s} {'Solved':>12s} {'Absolute difference':>20s}")
linear_errors = []
for index, name in enumerate(COEFFICIENTS):
    estimate = original_coefficients[index]
    estimate += derivatives['alpha'][f'd{name}_dalpha'] * np.deg2rad(increments_deg['alpha'])
    estimate += derivatives['beta'][f'd{name}_dbeta'] * np.deg2rad(increments_deg['beta'])
    estimate += derivatives['controls']['wing_section_1'][f'd{name}_ddelta'] * np.deg2rad(increments_deg['delta'])
    actual = getattr(combined, name)
    linear_errors.append(abs(actual - estimate))
    print(f'{name:12s} {estimate:+12.7f} {actual:+12.7f} {abs(actual - estimate):20.3e}')
assert np.isfinite(linear_errors).all()

Coefficient        Linear       Solved  Absolute difference
CX             -0.0016054   -0.0016448            3.935e-05
CY             +0.0000041   +0.0000057            1.613e-06
CZ             +0.1947598   +0.1947680            8.159e-06
Cl             -0.0000000   -0.0000004            3.751e-07
Cm             +0.0368309   +0.0368288            2.158e-06
Cn             -0.0000001   -0.0000001            3.642e-08


### Visualizing Local Slopes

Each solid curve below comes from separately solved perturbations, while the dashed line is the tangent predicted from the baseline derivative. The horizontal axis is the **increment** from the baseline in degrees; the tangent calculation converts it to radians. The curves are not independent experimental data: they visualize the same solver, holding the other inputs fixed. Asymmetry between the solid curve and tangent indicates local nonlinear or numerical effects over the chosen range, not a measured uncertainty.

In [10]:
offsets_deg = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
response_cases = {
    parameter: [perturbed_case(parameter, float(offset)) for offset in offsets_deg]
    for parameter in ('alpha', 'beta', 'delta')
}
fig_responses = make_subplots(
    rows=3, cols=2,
    subplot_titles=('CZ versus Δα', 'Cm versus Δα', 'CY versus Δβ',
                    'Cn versus Δβ', 'CZ versus Δδ', 'Cm versus Δδ'),
)
response_definitions = (
    ('alpha', 'CZ'), ('alpha', 'Cm'), ('beta', 'CY'),
    ('beta', 'Cn'), ('delta', 'CZ'), ('delta', 'Cm'),
)
for index, (parameter, coefficient) in enumerate(response_definitions):
    row, column = divmod(index, 2)
    api_block = derivatives['controls']['wing_section_1'] if parameter == 'delta' else derivatives[parameter]
    derivative = api_block[f'd{coefficient}_d{parameter}']
    baseline_value = original_coefficients[COEFFICIENTS.index(coefficient)]
    actual = np.array([getattr(solved, coefficient) for solved in response_cases[parameter]])
    tangent = baseline_value + derivative * np.deg2rad(offsets_deg)
    assert np.isfinite(actual).all() and np.isclose(actual[2], baseline_value, atol=1e-12, rtol=1e-12)
    fig_responses.add_trace(go.Scatter(x=offsets_deg, y=actual, mode='lines+markers', name='VLM solve',
                                       legendgroup='solve', showlegend=index == 0), row=row + 1, col=column + 1)
    fig_responses.add_trace(go.Scatter(x=offsets_deg, y=tangent, mode='lines', name='Local tangent',
                                       line={'dash': 'dash'}, legendgroup='tangent', showlegend=index == 0), row=row + 1, col=column + 1)
    fig_responses.update_xaxes(title_text=f'Δ{parameter} [degrees]', row=row + 1, col=column + 1)
    fig_responses.update_yaxes(title_text=coefficient, row=row + 1, col=column + 1)
fig_responses.update_layout(title='Coefficient response and local derivatives', height=900)
fig_responses.show()

### Visualizing Moment-Reference Sensitivity

The existing alpha sweep can be used to compare `Cm` about the solver's recalculated reference point with the same force and moment translated to the fixed baseline coordinate defined earlier. Both curves use the same VLM solution; their difference isolates the reference translation for these states. Neither curve is automatically a CG-based stability result.

In [ ]:
moving_moments = np.array([solved.Cm for solved in response_cases['alpha']])
fixed_moments = np.array([
    pitching_coefficient_at_fixed_point(solved, fixed_point)
    for solved in response_cases['alpha']
])
assert np.isfinite(moving_moments).all() and np.isfinite(fixed_moments).all()
assert np.isclose(moving_moments[2], fixed_moments[2], rtol=1e-12, atol=1e-12)
fig_moment_reference = go.Figure()
fig_moment_reference.add_trace(go.Scatter(x=offsets_deg, y=moving_moments, mode='lines+markers', name='Solver reference'))
fig_moment_reference.add_trace(go.Scatter(x=offsets_deg, y=fixed_moments, mode='lines+markers', name='Fixed baseline coordinate', line={'dash': 'dash'}))
fig_moment_reference.update_layout(
    title='Pitching coefficient and moment reference',
    xaxis_title='Angle-of-attack increment Δα [degrees]', yaxis_title='Cm',
    template='plotly_white',
)
fig_moment_reference.show()

## Verification and Validation

**Verification** asks whether the implementation solves its stated numerical problem and computes the reported finite differences consistently. We check the linear residual, all 18 API derivatives against separately solved perturbations, symmetry, baseline restoration, and sensitivity to the perturbation step. The thresholds below are numerical regression checks for this particular case, not physical accuracy limits. The condition number is reported as a diagnostic, not used to certify conditioning.

**Theoretical comparison** uses an independently assembled Prandtl lifting-line solution for the same rectangular planform and section slope $a_0=2\pi$ rad$^{-1}$. With $y=-b\cos\theta/2$, the Fourier coefficients solve

$$\sum_{k=1}^{N} A_k\sin(k\theta_i)\left(\frac{4b}{a_0c}+\frac{k}{\sin\theta_i}\right)=\alpha,\qquad C_L=\pi AR A_1.$$

We compare the scalar $dC_L/d\alpha$ against this independent model, and compare 40 and 60 Fourier modes to check the reference discretization. Lifting-line theory and VLM differ in formulation and mesh; this is a **model-to-model comparison**, not experimental validation. The reference does not validate $C_m$, $\beta$, or control derivatives. Its assumptions are discussed in [Leishman, Lifting Line & Finite Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/), equations 35 and 89–92. Experimental validation requires matched measured data and uncertainty, which are not included here.

In [11]:
def lifting_line_slope(span, chord, modes=40, section_slope=2 * np.pi):
    if span <= 0 or chord <= 0 or modes < 1 or section_slope <= 0:
        raise ValueError('Span, chord, mode count, and section slope must be positive.')
    theta = np.arange(1, modes + 1) * np.pi / (modes + 1)
    indices = np.arange(1, modes + 1)
    matrix = np.sin(theta[:, None] * indices[None, :]) * (
        4 * span / (section_slope * chord) + indices[None, :] / np.sin(theta[:, None])
    )
    coefficients = np.linalg.solve(matrix, np.ones(modes))
    return np.pi * span / chord * coefficients[0]


theory_slope_40 = lifting_line_slope(reference['span'], 1.0, modes=40)
theory_slope_60 = lifting_line_slope(reference['span'], 1.0, modes=60)
theory_mode_difference = abs(theory_slope_40 - theory_slope_60) / abs(theory_slope_60)
theory_difference = abs(scalar_cl_slope - theory_slope_60) / abs(theory_slope_60)
linear_residual = np.linalg.norm(case.P_ij @ case.gammas - case.w_i) / max(np.linalg.norm(case.w_i), np.finfo(float).eps)
condition_number = np.linalg.cond(case.P_ij)
api_discrepancies = []
for parameter in ('alpha', 'beta', 'delta'):
    block = derivatives['controls']['wing_section_1'] if parameter == 'delta' else derivatives[parameter]
    api_values = np.array([block[f'd{name}_d{parameter}'] for name in COEFFICIENTS])
    api_discrepancies.append(np.max(np.abs(api_values - finite_difference(parameter))))
max_api_discrepancy = max(api_discrepancies)
step_difference = abs(step_study[1]['dCZ_dalpha'] - step_study[2]['dCZ_dalpha']) / abs(step_study[2]['dCZ_dalpha'])
checks = [
    ('Linear-system relative residual', linear_residual, 1e-10),
    ('Maximum API/direct difference [rad⁻¹]', max_api_discrepancy, 1e-10),
    ('Maximum symmetry component [rad⁻¹]', max_symmetry_error, 1e-10),
    ('Relative dCZ/dalpha step difference', step_difference, 5e-5),
    ('Relative reference mode difference', theory_mode_difference, 1e-4),
]
print(f"{'Verification':44s} {'Value':>12s} {'Limit':>12s} Status")
for name, value, limit in checks:
    status = 'PASS' if np.isfinite(value) and value <= limit else 'FAIL'
    print(f'{name:44s} {value:12.3e} {limit:12.3e} {status}')
assert all(np.isfinite(value) and value <= limit for _, value, limit in checks)
assert case.export_configuration(include_flight=True) == original_configuration
print(f'Influence-matrix condition number: {condition_number:.3e}')
print(f'Lifting-line reference slope (60 modes): {theory_slope_60:.6f} rad⁻¹')
print(f'VLMPy scalar dCL/dalpha: {scalar_cl_slope:.6f} rad⁻¹')
print(f'Difference from lifting-line reference: {theory_difference:.2%} (informational)')

Verification                                        Value        Limit Status
Linear-system relative residual                 7.035e-16    1.000e-10 PASS
Maximum API/direct difference [rad⁻¹]           0.000e+00    1.000e-10 PASS
Maximum symmetry component [rad⁻¹]              6.809e-12    1.000e-10 PASS
Relative dCZ/dalpha step difference             1.259e-06    5.000e-05 PASS
Relative reference mode difference              9.095e-07    1.000e-04 PASS
Influence-matrix condition number: 4.788e+00
Lifting-line reference slope (60 modes): 4.837703 rad⁻¹
VLMPy scalar dCL/dalpha: 4.760686 rad⁻¹
Difference from lifting-line reference: 1.59% (informational)


In [12]:
spanwise_counts = [row['n'] for row in mesh_study]
mesh_cl_slopes = []
for span_panels in spanwise_counts:
    plus = perturbed_case('alpha', 0.1, n=span_panels, m=4)
    minus = perturbed_case('alpha', -0.1, n=span_panels, m=4)
    mesh_cl_slopes.append((plus.CL - minus.CL) / (2 * np.deg2rad(0.1)))
assert np.isclose(mesh_cl_slopes[1], scalar_cl_slope, rtol=1e-10, atol=1e-10)
model_discrepancy_percent = 100 * np.abs(np.asarray(mesh_cl_slopes) - theory_slope_60) / abs(theory_slope_60)
fig_validation = make_subplots(rows=1, cols=2, subplot_titles=('Lift slope comparison', 'Difference from lifting-line model'))
fig_validation.add_trace(go.Scatter(x=spanwise_counts, y=mesh_cl_slopes, mode='lines+markers', name='VLMPy scalar dCL/dα'), row=1, col=1)
fig_validation.add_trace(go.Scatter(x=spanwise_counts, y=[theory_slope_60] * len(spanwise_counts), mode='lines', name='Lifting-line reference', line={'dash': 'dash'}), row=1, col=1)
fig_validation.add_trace(go.Scatter(x=spanwise_counts, y=model_discrepancy_percent, mode='lines+markers', name='Model difference'), row=1, col=2)
fig_validation.update_xaxes(title_text='Spanwise panels n')
fig_validation.update_yaxes(title_text='dCL/dα [rad⁻¹]', row=1, col=1)
fig_validation.update_yaxes(title_text='Relative difference [%]', row=1, col=2)
fig_validation.update_layout(title='Theoretical comparison at different spanwise meshes', height=420)
fig_validation.show()

### Independent Chordwise Refinement

The existing mesh study varies spanwise count while holding $m=4$. Here, hold $n=16$, flight condition, and finite-difference step fixed while varying chordwise count. Report requested and effective mesh sizes and all three derivative families. Non-monotonic values are evidence that further refinement and inspection of the control-boundary representation are needed, not grounds for declaring convergence.

In [ ]:
chordwise_study = []
for chord_panels in (2, 4, 8):
    chord_case = solve_case(wing, n=16, m=chord_panels)
    with contextlib.redirect_stdout(io.StringIO()):
        value = chord_case.calculate_static_derivatives(step_deg=0.1)
    row = {
        'n_requested': 16, 'm_requested': chord_panels,
        'n_effective': chord_case.n, 'm_effective': chord_case.m,
        'dCZ_dalpha': value['alpha']['dCZ_dalpha'],
        'dCm_dalpha': value['alpha']['dCm_dalpha'],
        'dCZ_ddelta': value['controls']['wing_section_1']['dCZ_ddelta'],
    }
    assert np.isfinite(list(row.values())).all()
    chordwise_study.append(row)
    print(f"requested (n,m)=({row['n_requested']},{row['m_requested']}), effective ({row['n_effective']},{row['m_effective']}): "
          f"dCZ/dalpha={row['dCZ_dalpha']:+.6f}, dCm/dalpha={row['dCm_dalpha']:+.6f}, dCZ/ddelta={row['dCZ_ddelta']:+.6f} rad⁻¹")

fig_chordwise = go.Figure()
for name, label in (('dCZ_dalpha', 'dCZ/dalpha'), ('dCm_dalpha', 'dCm/dalpha'), ('dCZ_ddelta', 'dCZ/ddelta')):
    fig_chordwise.add_trace(go.Scatter(x=[row['m_effective'] for row in chordwise_study],
        y=[row[name] for row in chordwise_study], mode='lines+markers', name=label))
fig_chordwise.update_layout(title='Static derivative sensitivity to chordwise mesh',
    xaxis_title='Effective chordwise panels m', yaxis_title='Derivative [rad⁻¹]', template='plotly_white')
fig_chordwise.show()

## 9. API Boundaries and Next Steps

`step_deg` must be positive and finite. With no wing-section `deflection_angle`, the API should return an empty `controls` mapping; the next cell verifies both cases. A section with a deflection key needs a complete, valid control geometry when the solver builds the mesh.

For broader validation, compare matched geometry, operating point, wake assumptions, axes, signs, normalization, fixed moment point, and controls against an independent lifting-surface code or a documented wind-tunnel dataset. Report mesh and step studies, Reynolds and Mach conditions, uncertainty, and any viscous effects before interpreting discrepancies. This notebook performs numerical verification and plausibility checks only; it does not establish experimental validity or certify aircraft stability.

In [9]:
for invalid_step in (0.0, -0.1, np.nan, np.inf):
    try:
        case.calculate_static_derivatives(step_deg=invalid_step)
    except ValueError:
        pass
    else:
        raise AssertionError(f'Invalid step accepted: {invalid_step}')

plain_wing = copy.deepcopy(wing)
for key in ('flap_start', 'flap_end', 'flap_hinge_chord', 'deflection_angle', 'deflection_type'):
    plain_wing['wing_sections'][0].pop(key)
plain_case = solve_case(plain_wing)
with contextlib.redirect_stdout(io.StringIO()):
    no_control_derivatives = plain_case.calculate_static_derivatives(step_deg=0.1)
assert no_control_derivatives['controls'] == {}
assert case.export_configuration(include_flight=True) == original_configuration
print('Invalid step sizes rejected; unconfigured controls omitted; baseline unchanged.')

Invalid step sizes rejected; unconfigured controls omitted; baseline unchanged.


## Verification and Validation

Separate-case finite differences, symmetry, state restoration, and step/mesh studies check numerical behavior. The lifting-line comparison covers only scalar lift slope; no experimental validation of lateral, control, or moment derivatives is included.

## References

- [Leishman: Lifting-Line and Finite-Wing Theory](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/lifting-line-theory/).
- [NASA Glenn: Aircraft Rotations](https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/aircraft-rotations/) and [Aircraft Flight Mechanics: Longitudinal Stability](https://aircraftflightmechanics.com/StaticStability/LSS.html).

## Use Cases

Use these local derivatives to inspect the sensitivity of the implemented steady aerodynamic model near a specified configuration, not to infer aircraft stability without matched conventions and trim.

## Possible Improvements

Add a fixed-CG interface and compare against a published derivative dataset or another validated solver with matched geometry, units, axes, wake assumptions, and controls. Study chordwise convergence and the observed control-mesh sensitivity.

## Next Steps

Use the [coefficient notebook](06_aerodynamic_coefficients.ipynb) to inspect the loads and moment references before exporting derivatives to any future flight-dynamics model.